In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline


In [ ]:

# =========================
# 1) Load prepared numeric data
# =========================
df = pd.read_csv("../fraud_preprocessing/fraud_prepared_numeric.csv")

target = "fraud"
X = df.drop(columns=[target])
y = df[target].astype(int)


In [ ]:

# =========================
# 2) Split columns:
#    - binary columns: keep as-is
#    - continuous columns: scale
# =========================
binary_cols = [c for c in X.columns if set(X[c].dropna().unique()).issubset({0, 1})]
continuous_cols = [c for c in X.columns if c not in binary_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), continuous_cols),
        ("binary", "passthrough", binary_cols),
    ],
    remainder="drop"
)


In [ ]:

# =========================
# 3) Base model for wrapper selection
#    class_weight='balanced' is important for fraud imbalance
# =========================
base_model = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
    random_state=42
)


In [ ]:

# =========================
# 4) RFECV: recursive feature elimination with CV
#    scoring='average_precision' is better than accuracy for fraud problems
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

selector = RFECV(
    estimator=base_model,
    step=2,                       # remove 2 features at a time
    cv=cv,
    scoring="average_precision",  # better for imbalanced fraud label
    min_features_to_select=5,
    n_jobs=-1
)

pipe = Pipeline([
    ("prep", preprocess),
    ("rfecv", selector)
])

# Fit selector
pipe.fit(X, y)


In [ ]:

# =========================
# 5) Recover selected feature names
# =========================
feature_names_after_prep = pipe.named_steps["prep"].get_feature_names_out()
support_mask = pipe.named_steps["rfecv"].support_
ranking = pipe.named_steps["rfecv"].ranking_

selected_features = [
    name.replace("scale__", "").replace("binary__", "")
    for name, keep in zip(feature_names_after_prep, support_mask)
    if keep
]

ranking_df = pd.DataFrame({
    "feature": [
        name.replace("scale__", "").replace("binary__", "")
        for name in feature_names_after_prep
    ],
    "rank": ranking
}).sort_values(["rank", "feature"])

# Remove duplicates just in case
selected_features = list(dict.fromkeys(selected_features))


In [ ]:

# =========================
# 6) Save reduced dataset
# =========================
OUT_DIR = Path("../wrapper")
OUT_DIR.mkdir(parents=True, exist_ok=True)

selected_df = df[selected_features + [target]].copy()
selected_df.to_csv("../wrapper/fraud_selected_features_rfecv.csv", index=False)


In [ ]:

# =========================
# 7) Print results
# =========================
print("Original number of features :", X.shape[1])
print("Selected number of features :", len(selected_features))
print("\nSelected features:")
for f in selected_features:
    print("-", f)

print("\nTop-ranked features:")
print(ranking_df.head(30).to_string(index=False))

print("\nReduced dataset saved as: fraud_selected_features_rfecv.csv")